In [6]:
%pip install -U langchain-google-genai

In [7]:
# Import library getpass
# Digunakan untuk memasukkan API key secara aman
import getpass

# Import library os
# Digunakan untuk mengakses environment variable pada sistem
import os

# Mengecek apakah GOOGLE_API_KEY sudah ada di environment variable
# Jika belum ada, maka program akan meminta user memasukkan API key Gemini
if "GOOGLE_API_KEY" not in os.environ:

    # Menyimpan API key yang diinput user ke environment variable
    # getpass() membuat input tidak terlihat saat diketik
    os.environ["GOOGLE_API_KEY"] = getpass.getpass(
        "Enter your Google AI API key: "
    )

Enter your Google AI API key: ··········


In [8]:
# Import ChatGoogleGenerativeAI dari library LangChain Google GenAI
# Digunakan untuk menghubungkan notebook dengan model Gemini dari Google
from langchain_google_genai import ChatGoogleGenerativeAI

# Membuat object LLM (Large Language Model) menggunakan Gemini
llm = ChatGoogleGenerativeAI(

    # Menentukan model Gemini yang digunakan
    model="gemini-3-flash-preview",

    # Mengatur tingkat kreativitas jawaban AI
    # temperature=0 membuat jawaban lebih konsisten dan faktual
    temperature=0,

    # Mengatur jumlah maksimum token output
    # None berarti menggunakan default dari model Gemini
    max_tokens=None,

    # Mengatur batas waktu request ke model
    # None berarti tidak ada timeout khusus
    timeout=None,

    # Mengatur jumlah percobaan ulang jika request gagal
    # Jika error sementara terjadi, sistem akan mencoba ulang maksimal 2 kali
    max_retries=2,
)

In [9]:
# Import library pandas
# Pandas digunakan untuk mengelola dataset
import pandas as pd

# Membaca file CSV dataset rumah
# "/content/data_rumah.csv" adalah lokasi file di Google Colab
# sep=';' digunakan karena file CSV menggunakan pemisah titik koma (;)
df = pd.read_csv("/content/data_rumah.csv", sep=';')

# Menampilkan 5 data pertama dari dataset
df.head()

,NO,NAMA RUMAH,HARGA,LB,LT,KT,KM,GRS
0,1,"Rumah Murah Hook Tebet Timur, Tebet, Jakarta S...",3800000000,220,220,3,3,0
1,2,"Rumah Modern di Tebet dekat Stasiun, Tebet, Ja...",4600000000,180,137,4,3,2
2,3,"Rumah Mewah 2 Lantai Hanya 3 Menit Ke Tebet, T...",3000000000,267,250,4,4,4
3,4,"Rumah Baru Tebet, Tebet, Jakarta Selatan",430000000,40,25,2,2,0
4,5,"Rumah Bagus Tebet komp Gudang Peluru lt 350m, ...",9000000000,400,355,6,5,3


In [10]:
# Membuat list kosong bernama documents
# List ini akan digunakan untuk menyimpan seluruh dokumen teks
# yang nantinya akan diubah menjadi embedding dan dimasukkan ke vector database
documents = []

# Melakukan iterasi/perulangan untuk membaca setiap baris pada dataset
# df.iterrows() digunakan untuk mengambil data per baris
for _, row in df.iterrows():

    # Mengubah setiap baris data rumah menjadi teks natural language
    # Tujuannya agar embedding model lebih mudah memahami isi data
    text = f"""
    {row['NAMA RUMAH']}.

    Harga rumah sebesar {row['HARGA']} rupiah.
    Luas bangunan {row['LB']} meter persegi.
    Luas tanah {row['LT']} meter persegi.
    Memiliki {row['KT']} kamar tidur,
    {row['KM']} kamar mandi,
    dan garasi {row['GRS']} mobil.
    """

    # Menambahkan text document ke dalam list documents untuk proses embedding
    documents.append(text)

In [13]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install langchain-community

In [14]:
# Import HuggingFaceEmbeddings dari LangChain Community
# Digunakan untuk mengubah teks menjadi embedding/vector numerik
from langchain_community.embeddings import HuggingFaceEmbeddings

# Import FAISS vector database
# FAISS digunakan untuk menyimpan embedding dan melakukan
# similarity search atau pencarian dokumen yang paling relevan
from langchain_community.vectorstores import FAISS

# Membuat model embedding menggunakan HuggingFace
embedding_model = HuggingFaceEmbeddings(

    # Menggunakan model embedding all-MiniLM-L6-v2
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
# Membuat vector database menggunakan FAISS
# FAISS.from_texts() digunakan untuk:
# 1. Mengubah seluruh dokumen teks menjadi embedding/vector
# 2. Menyimpan embedding tersebut ke dalam vector database FAISS
vectorstore = FAISS.from_texts(

    # documents berisi kumpulan text document hasil preprocessing dataset rumah
    # Data inilah yang akan diubah menjadi embedding
    documents,

    # embedding_model digunakan untuk mengubah text menjadi vector numerik
    # menggunakan model sentence-transformers/all-MiniLM-L6-v2
    embedding_model
)

In [16]:
# Mengubah vector database menjadi retriever
# Retriever digunakan sebagai tool RAG untuk mencari dokumen
retriever = vectorstore.as_retriever()

In [18]:
# Membuat pertanyaan/query yang akan diberikan ke sistem RAG
query = "Rumah dengan 4 kamar tidur"

# Menggunakan retriever untuk mencari dokumen yang paling relevan
# berdasarkan query menggunakan semantic similarity search
docs = retriever.invoke(query)

# Menampilkan isi dokumen pertama yang paling relevan
# page_content berisi text document hasil retrieval dari vector database
docs[0].page_content

'\n    Rumah Tebet Ada Kamar Tidur.\n\n    Harga rumah sebesar 3700000000 rupiah.\n    Luas bangunan 400 meter persegi.\n    Luas tanah 370 meter persegi.\n    Memiliki 6 kamar tidur,\n    2 kamar mandi,\n    dan garasi 0 mobil.\n    '

In [21]:
# Menggabungkan seluruh dokumen hasil retrieval menjadi satu context
# "\n" digunakan untuk memberi baris baru antar dokumen
context = "\n".join([doc.page_content for doc in docs])

# Membuat prompt untuk diberikan ke model Gemini
# Prompt berisi:
# 1. Instruksi
# 2. Context hasil retrieval dari vector database
# 3. Pertanyaan user
prompt = f"""
Jawab pertanyaan berdasarkan context berikut.

Context:
{context}

Question:
{query}
"""

# Mengirim prompt ke model Gemini menggunakan method invoke()
# Model akan menghasilkan jawaban berdasarkan context RAG yang diberikan
response = llm.invoke(prompt)

# Menampilkan jawaban dari model Gemini
# response.text berisi hasil jawaban akhir dari sistem RAG
print(response.text)

Berdasarkan context yang diberikan, rumah dengan **4 kamar tidur** adalah:

**Rumah Area Tebet Ada Kamar Tidur**
*   **Harga:** 3.000.000.000 rupiah
*   **Luas Bangunan:** 192 meter persegi
*   **Luas Tanah:** 120 meter persegi
*   **Kamar Tidur:** 4
*   **Kamar Mandi:** 3
*   **Garasi:** 2 mobil


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


In [22]:
# Menampilkan judul section source documents
# Bagian ini digunakan untuk menunjukkan referensi hasil retrieval dari RAG
print("\n===== SOURCE DOCUMENTS =====\n")

# Melakukan perulangan untuk menampilkan seluruh dokumen hasil retrieval
# enumerate() digunakan untuk memberikan nomor pada setiap dokumen
for i, doc in enumerate(docs):

    # Menampilkan nomor dokumen hasil retrieval
    print(f"Document {i+1}:")

    # Menampilkan isi dokumen yang ditemukan oleh retriever
    # page_content berisi text document hasil similarity search dari vector database
    print(doc.page_content)

    # Menampilkan garis pemisah agar output lebih rapi dan mudah dibaca
    print("-" * 50)


===== SOURCE DOCUMENTS =====

Document 1:

    Rumah Tebet Ada Kamar Tidur.

    Harga rumah sebesar 3700000000 rupiah.
    Luas bangunan 400 meter persegi.
    Luas tanah 370 meter persegi.
    Memiliki 6 kamar tidur,
    2 kamar mandi,
    dan garasi 0 mobil.
    
--------------------------------------------------
Document 2:

    Rumah Area Tebet Ada Kamar Tidur.

    Harga rumah sebesar 3000000000 rupiah.
    Luas bangunan 192 meter persegi.
    Luas tanah 120 meter persegi.
    Memiliki 4 kamar tidur,
    3 kamar mandi,
    dan garasi 2 mobil.
    
--------------------------------------------------
Document 3:

    Rumah gang kebun baru tebet jakarta selatan.

    Harga rumah sebesar 900000000 rupiah.
    Luas bangunan 100 meter persegi.
    Luas tanah 56 meter persegi.
    Memiliki 3 kamar tidur,
    2 kamar mandi,
    dan garasi 0 mobil.
    
--------------------------------------------------
Document 4:

    Rumah Tebet Dijual Cepat Dengan Lokasi Strategis.

    Harga rumah se